In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print(f'pandas  {pd.__version__}')
print(f'numpy   {np.__version__}')

pandas  3.0.0
numpy   2.4.1


In [2]:
OUTCOMES_URL = 'https://data.austintexas.gov/resource/gsvs-ypi7.csv'
INTAKES_URL  = 'https://data.austintexas.gov/resource/pyqf-r2dc.csv'
PAGE_SIZE    = 50_000  # Socrata max rows per request; without this we silently get 1,000

# Local caches (delete to refetch from API)
RAW_PATH         = Path('../data/new_system_data/austin_animal_outcomes.csv')
RAW_INTAKES_PATH = Path('../data/new_system_data/austin_animal_intakes.csv')

PROCESSED_PATH   = Path('../data/processed/outcomes_clean.csv')

DOMESTIC_TYPES = {'Cat', 'Dog', 'Kitten', 'Puppy'}

OUTCOME_MAP = {
    'Adopted':                    'Adopted',
    'Adopted Altered':            'Adopted',
    'Adopted Unaltered':          'Adopted',
    'Adopted Offsite':            'Adopted',
    'Adopted Offsite(Altered)':   'Adopted',
    'Adopted Offsite(Unaltered)': 'Adopted',
    'Adopted to Finder':          'Adopted',
    'Adoption':                   'Adopted',
    'Transfer Out':               'Transferred',
    'Transfer':                   'Transferred',
    'Foster':                     'Transferred',
    'Reclaimed':                  'Reclaimed',
    'Redemption (Offsite)':       'Reclaimed',
    'Return to Owner':            'Reclaimed',
    'Returned To Owner':          'Reclaimed',
    'Rto-Adopt':                  'Reclaimed',
    'Euthanized':                 'Euthanized',
    'Euthanasia':                 'Euthanized',
    'Euthanasia - RTO':           'Euthanized',
    'Died':                       'Died',
    'Unassisted Death':           'Died',
    'Unassisted Death - In Transit': 'Died',
    'Unassisted Death - In Foster':  'Died',
    'DOA':                        'Died',
    'DOA - Released to Owner':    'Died',
    'DOA - Final Disposition':    'Died',
    'Disposal':                   'Died',
    'Interred':                   'Died',
    'Cremated':                   'Died',
    'Cremation - Communal':       'Died',
    'Cremation - Private':        'Died',
}
# Remaining values -> 'Other': Unresolved File, Escaped, Stolen, Bite Quarantine (Home)

## 2 · Data Loading

Both datasets are fetched via the Socrata API and cached locally so re-runs do not
re-download. The shared `fetch_all_records()` helper paginates through all records.

### 2a · Outcomes and intakes Dataset (After 05/05/2025)

In [3]:
from datetime import date


def fetch_all_records(base_url: str, page_size: int = 50_000) -> pd.DataFrame:
    frames, offset = [], 0
    while True:
        chunk = pd.read_csv(f'{base_url}?$limit={page_size}&$offset={offset}')
        if chunk.empty:
            break
        frames.append(chunk)
        print(f'  fetched rows {offset:>7,} - {offset + len(chunk):>7,}')
        if len(chunk) < page_size:
            break
        offset += page_size
    return pd.concat(frames, ignore_index=True)


def cache_is_fresh(path: Path) -> bool:
    # Cache is reused only if it was written today; otherwise refetch from API.
    return path.exists() and date.fromtimestamp(path.stat().st_mtime) == date.today()


if cache_is_fresh(RAW_PATH):
    df_outcomes_raw = pd.read_csv(RAW_PATH)
    print(f'Loaded from cache: {df_outcomes_raw.shape[0]:,} rows x {df_outcomes_raw.shape[1]} columns')
    print(f'(Cache dated {date.today()} — delete {RAW_PATH} to force refetch)')
else:
    if RAW_PATH.exists():
        print(f'Cache is stale (mtime != {date.today()}) — refetching from API...')
    else:
        print('Fetching all outcomes from Socrata API...')
    df_outcomes_raw = fetch_all_records(OUTCOMES_URL, PAGE_SIZE)
    RAW_PATH.parent.mkdir(parents=True, exist_ok=True)
    df_outcomes_raw.to_csv(RAW_PATH, index=False)
    print(f'Loaded: {df_outcomes_raw.shape[0]:,} rows x {df_outcomes_raw.shape[1]} columns')
    print(f'Cached -> {RAW_PATH}')

Loaded from cache: 12,967 rows x 16 columns
(Cache dated 2026-05-29 — delete ../data/new_system_data/austin_animal_outcomes.csv to force refetch)


### 2b · Intakes Dataset (After 05/05/2025)

Records each animal's arrival at the shelter: intake type (Stray, Owner Surrender,
Public Assist, etc.), intake condition, and intake date.
Will be joined to outcomes on `animal_id` in a later notebook.

In [4]:
if cache_is_fresh(RAW_INTAKES_PATH):
    df_intakes_raw = pd.read_csv(RAW_INTAKES_PATH)
    print(f'Loaded from cache: {df_intakes_raw.shape[0]:,} rows x {df_intakes_raw.shape[1]} columns')
    print(f'(Cache dated {date.today()} — delete {RAW_INTAKES_PATH} to force refetch)')
else:
    if RAW_INTAKES_PATH.exists():
        print(f'Cache is stale (mtime != {date.today()}) — refetching from API...')
    else:
        print('Fetching all intakes from Socrata API...')
    df_intakes_raw = fetch_all_records(INTAKES_URL, PAGE_SIZE)
    RAW_INTAKES_PATH.parent.mkdir(parents=True, exist_ok=True)
    df_intakes_raw.to_csv(RAW_INTAKES_PATH, index=False)
    print(f'Loaded: {df_intakes_raw.shape[0]:,} rows x {df_intakes_raw.shape[1]} columns')
    print(f'Cached -> {RAW_INTAKES_PATH}')

Loaded from cache: 13,035 rows x 15 columns
(Cache dated 2026-05-29 — delete ../data/new_system_data/austin_animal_intakes.csv to force refetch)
